# Week 4: Retrieval-Augmented Generation (RAG) with arXiv Papers
This week marks a major shift in your AI agent's capabilities: you'll build the foundation for a Retrieval-Augmented Generation (RAG) system tailored to scientific research. Rather than relying on an LLM's memory alone, RAG architectures allow your agent to search a structured knowledge base and generate grounded, document-aware answers.

Your task is to create a RAG pipeline using recent arXiv cs.CL papers, converting them into searchable chunks, embedding them, and indexing them with FAISS. You'll then implement a simple query interface that takes a user question, retrieves the top relevant chunks, and displays them for further processing.

This week marks the beginning of building your agent's private research knowledge base—a semantic index that you'll evolve into a full-featured hybrid database in Week 5.

## 📚 Learning Objectives

* Understand the components of a Retriever-Reader QA pipeline.
* Explore document chunking strategies (e.g., sections vs. sliding windows) and their impact on retrieval performance.
* Index scientific text using vector embeddings and FAISS.
* Build and query a semantic index via a FastAPI endpoint that returns relevant passages.

## 项目实现

下面是完整的RAG系统实现，包括数据获取、处理、索引构建和查询功能。

### 1. 安装依赖包

In [9]:
# 安装必要的包
!pip install PyMuPDF sentence-transformers faiss-cpu arxiv tqdm -q

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


### 2. 导入必要的库

In [20]:
import fitz  # PyMuPDF
import numpy as np
import faiss
import json
import os
from typing import List, Dict
from sentence_transformers import SentenceTransformer
from pathlib import Path
from tqdm import tqdm
import arxiv
import time

### 3. 下载arXiv论文（示例：下载50篇论文）

In [22]:
def download_sample_papers(num_papers=10):
    """下载示例论文"""
    output_path = Path("papers")
    output_path.mkdir(exist_ok=True)
    
    # 搜索cs.CL分类的论文
    search = arxiv.Search(
        query="cat:cs.CL",
        max_results=num_papers,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending
    )
    
    papers_info = []
    downloaded = 0
    
    for result in tqdm(search.results(), total=num_papers, desc="Downloading papers"):
        try:
            # 清理文件名
            safe_title = "".join(c for c in result.title if c.isalnum() or c in (' ', '-', '_')).rstrip()[:50]
            filename = f"{safe_title}.pdf"
            filepath = output_path / filename
            
            # 如果文件不存在则下载
            if not filepath.exists():
                result.download_pdf(dirpath=str(output_path), filename=filename)
                time.sleep(0.5)  # 避免请求过快
            
            papers_info.append({
                'title': result.title,
                'filename': filename,
                'abstract': result.summary[:500]
            })
            downloaded += 1
            
        except Exception as e:
            print(f"Error downloading {result.title}: {e}")
    
    print(f"Downloaded {downloaded} papers to 'papers' directory")
    return papers_info

# 下载论文
papers_info = download_sample_papers(50)

/var/folders/md/jx00px7j64jb171dzf32jybr0000gn/T/ipykernel_52543/1529618288.py:17: DeprecationWarning: The 'Search.results' method is deprecated, use 'Client.results' instead
  for result in tqdm(search.results(), total=num_papers, desc="Downloading papers"):


KeyboardInterrupt: 

### 4. 实现RAG管道类

In [12]:
class RAGPipeline:
    def __init__(self, model_name='all-MiniLM-L6-v2'):
        """初始化RAG管道"""
        print(f"Loading embedding model: {model_name}")
        self.model = SentenceTransformer(model_name)
        self.chunks = []
        self.chunk_metadata = []
        self.index = None
        
    def extract_text_from_pdf(self, pdf_path: str) -> str:
        """从PDF提取文本"""
        try:
            doc = fitz.open(pdf_path)
            pages = []
            for page in doc:
                page_text = page.get_text().strip()
                if page_text:
                    pages.append(page_text)
            doc.close()
            return "\n".join(pages)
        except Exception as e:
            print(f"Error extracting {pdf_path}: {e}")
            return ""
    
    def chunk_text(self, text: str, max_tokens: int = 512, overlap: int = 50) -> List[str]:
        """将文本分割成块"""
        words = text.split()
        chunks = []
        step = max_tokens - overlap
        
        for i in range(0, len(words), step):
            chunk = " ".join(words[i:i + max_tokens])
            chunks.append(chunk)
            if i + max_tokens >= len(words):
                break
        return chunks
    
    def process_papers(self, pdf_folder: str):
        """处理所有PDF论文"""
        pdf_files = list(Path(pdf_folder).glob("*.pdf"))
        print(f"Processing {len(pdf_files)} PDF files...")
        
        for pdf_path in tqdm(pdf_files, desc="Extracting text"):
            text = self.extract_text_from_pdf(str(pdf_path))
            if not text:
                continue
            
            chunks = self.chunk_text(text)
            for i, chunk in enumerate(chunks):
                self.chunks.append(chunk)
                self.chunk_metadata.append({
                    'file': pdf_path.name,
                    'chunk_id': i,
                    'total_chunks': len(chunks)
                })
        
        print(f"Created {len(self.chunks)} chunks")
    
    def create_embeddings(self):
        """创建嵌入向量"""
        print("Creating embeddings...")
        embeddings = self.model.encode(self.chunks, show_progress_bar=True)
        return embeddings
    
    def build_index(self, embeddings: np.ndarray):
        """构建FAISS索引"""
        dimension = embeddings.shape[1]
        self.index = faiss.IndexFlatL2(dimension)
        self.index.add(embeddings.astype('float32'))
        print(f"Built index with {self.index.ntotal} vectors")
    
    def search(self, query: str, k: int = 3) -> List[Dict]:
        """搜索相关文本块"""
        query_vector = self.model.encode([query])
        distances, indices = self.index.search(query_vector.astype('float32'), k)
        
        results = []
        for idx, dist in zip(indices[0], distances[0]):
            results.append({
                'chunk': self.chunks[idx],
                'metadata': self.chunk_metadata[idx],
                'distance': float(dist)
            })
        return results

### 5. 构建RAG索引

In [ ]:
# 创建RAG管道实例
rag = RAGPipeline()

# 处理PDF文件
rag.process_papers("papers")

# 创建嵌入向量
embeddings = rag.create_embeddings()

# 构建FAISS索引
rag.build_index(embeddings)

print(f"\n✅ RAG索引构建完成!")
print(f"📊 总文档块数: {len(rag.chunks)}")
print(f"📄 处理的论文数: {len(set(m['file'] for m in rag.chunk_metadata))}")

Loading embedding model: all-MiniLM-L6-v2
Processing 33 PDF files...


Extracting text: 100%|██████████| 33/33 [00:01<00:00, 21.64it/s]


Created 560 chunks
Creating embeddings...


Batches: 100%|██████████| 18/18 [00:01<00:00, 12.82it/s]

Built index with 560 vectors

✅ RAG索引构建完成!
📊 总文档块数: 560
📄 处理的论文数: 33


### 6. 测试搜索功能

In [ ]:
def test_search(rag_pipeline, queries):
    """测试搜索功能并展示结果"""
    for query in queries:
        print(f"\n{'='*80}")
        print(f"🔍 Query: {query}")
        print(f"{'='*80}")
        
        results = rag_pipeline.search(query, k=3)
        
        for i, result in enumerate(results, 1):
            print(f"\n📌 Result {i}:")
            print(f"📄 Source: {result['metadata']['file']}")
            print(f"📊 Distance: {result['distance']:.4f}")
            print(f"📝 Text snippet:")
            print(f"   {result['chunk'][:300]}...")
            print(f"   {'-'*70}")

# 测试查询
test_queries = [
    "What are transformers in NLP?",
    "How does attention mechanism work?",
    "What is BERT and how does it work?",
    "Explain language modeling techniques",
    "What is few-shot learning?"
]

test_search(rag, test_queries)


🔍 Query: What are transformers in NLP?

📌 Result 1:
📄 Source: Share Your Attention Transformer Weight Sharing via Matrix-based Dictionary Learning.pdf
📊 Distance: 0.9283
📝 Text snippet:
   2017. Using the Output Embedding to Improve Language Models. In Proceedings of the 15th Conference of the European Chapter of the Association for Computational Linguistics: Volume 2, Short Papers, 157– 163. Sun, S.; Cheng, Y.; Gan, Z.; and Liu, J. 2019. Patient Knowledge Distillation for BERT Model ...
   ----------------------------------------------------------------------

📌 Result 2:
📄 Source: Explainable Collaborative Problem Solving Diagnosis with BERT using SHAP and its Implications for Te.pdf
📊 Distance: 0.9628
📝 Text snippet:
   computer-mediated environments [5]. Several existing studies have investigated the use of Bidirectional Encoder Representations from Transformers (BERT) and its variant models for CPS diagnosis. For exam- ple, Pugh et al. [6] implemented a model (i.e., BERT-seq) tha

### 7. 保存索引和数据

In [ ]:
# 保存FAISS索引
output_dir = Path("rag_data")
output_dir.mkdir(exist_ok=True)

# 保存索引
faiss.write_index(rag.index, str(output_dir / "faiss_index.bin"))

# 保存文本块和元数据
with open(output_dir / "chunks.json", "w", encoding='utf-8') as f:
    json.dump(rag.chunks, f, ensure_ascii=False, indent=2)

with open(output_dir / "metadata.json", "w", encoding='utf-8') as f:
    json.dump(rag.chunk_metadata, f, ensure_ascii=False, indent=2)

print(f"✅ 索引和数据已保存到 {output_dir} 目录")

✅ 索引和数据已保存到 rag_data 目录


### 8. 交互式查询演示

In [16]:
def interactive_search(rag_pipeline):
    """交互式搜索界面"""
    print("📚 RAG Search System")
    print("Enter your query (or 'quit' to exit):\n")
    
    while True:
        query = input("\n🔍 Query: ").strip()
        
        if query.lower() in ['quit', 'exit', 'q']:
            print("Goodbye!")
            break
        
        if not query:
            continue
        
        results = rag_pipeline.search(query, k=3)
        
        print(f"\n📊 Found {len(results)} relevant passages:\n")
        
        for i, result in enumerate(results, 1):
            print(f"Result {i}:")
            print(f"  Source: {result['metadata']['file']}")
            print(f"  Relevance: {1/(1+result['distance']):.2%}")
            print(f"  Text: {result['chunk'][:200]}...")
            print()

# 运行交互式搜索（在Jupyter中可能需要调整）
# interactive_search(rag)

# 或者直接测试一个查询
query = "What are the latest advances in language models?"
print(f"Query: {query}\n")
results = rag.search(query, k=3)

for i, result in enumerate(results, 1):
    print(f"Result {i}:")
    print(f"  📄 File: {result['metadata']['file']}")
    print(f"  📊 Distance: {result['distance']:.4f}")
    print(f"  📝 Preview: {result['chunk'][:250]}...")
    print()

Query: What are the latest advances in language models?

Result 1:
  📄 File: AutoCodeBench Large Language Models are Automatic .pdf
  📊 Distance: 0.8494
  📝 Preview: Xuan-Son Nguyen, Cl´ementine Fourrier, Ben Burtenshaw, Hugo Larcher, Haojun Zhao, Cyril Zakka, Mathieu Morlon, Colin Raffel, Leandro von Werra, and Thomas Wolf. Smollm2: When smol goes big – data-centric training of a small language model, 2025. URL ...

Result 2:
  📄 File: Utilizing Multilingual Encoders to Improve Large L.pdf
  📊 Distance: 0.8550
  📝 Preview: S., Matena, M., Zhou, Y., Li, W., & Liu, P. J. (2019, October 23). Exploring the Limits of Transfer Learning with a Unified Text-to-Text Transformer. arXiv.org. https://arxiv.org/abs/1910.10683. [7] J. Kaplan et al., “Scaling laws for neural language...

Result 3:
  📄 File: CPO Addressing Reward Ambiguity in Role-playing Di.pdf
  📊 Distance: 0.8714
  📝 Preview: Evaluating character understanding of large language models via character profiling from fictional works. 